# NYC Taxi 실전 데이터 분석 - Spark DataFrame API

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- NYC TLC 공개 데이터를 다운로드하고 분석할 수 있다
- 실제 Parquet 파일에서 대용량 데이터를 읽을 수 있다
- 시간대별, 지역별 집계 분석을 수행할 수 있다
- Window Function으로 누적 매출, 이동 평균을 계산할 수 있다
- 조인을 활용하여 Zone 정보를 연결할 수 있다

---

## 실습 환경 안내

이 실습은 Docker 컨테이너(python-dev) 내에서 진행됩니다.

### 실행 방법 1: VSCode Dev Container (권장)

VSCode의 Dev Containers 익스텐션을 사용하면 컨테이너 내부에 직접 접속하여 작업할 수 있습니다.
- 왼쪽 하단에 `python-dev` 또는 컨테이너 이름이 표시됩니다.
- 터미널, 파일 탐색기 모두 컨테이너 내부에서 동작합니다.
- 이 화면에서 바로 Python 파일이나 노트북을 실행하면 됩니다.

### 실행 방법 2: docker exec

로컬 터미널에서 `docker exec` 명령어로 컨테이너 내부 명령을 실행할 수 있습니다.

```bash
# 컨테이너 내부에서 Python 스크립트 실행
docker exec -it python-dev python your_script.py

# 컨테이너 내부에서 대화형 셸 접속
docker exec -it python-dev bash
```

---

## NYC Taxi & Limousine Commission (TLC) 데이터

**공식 데이터 소스**: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

- 2009년부터 현재까지 뉴욕시 모든 택시 운행 기록
- 매월 수백만 ~ 수천만 건의 실제 운행 데이터
- Yellow Taxi, Green Taxi, FHV(Uber/Lyft) 데이터 제공
- Parquet 형식으로 제공 (효율적인 컬럼 기반 저장)

### Yellow Taxi 주요 컬럼

| 컬럼 | 설명 | 타입 |
|------|------|------|
| tpep_pickup_datetime | 승차 시각 | Timestamp |
| tpep_dropoff_datetime | 하차 시각 | Timestamp |
| PULocationID | 승차 구역 ID | Integer |
| DOLocationID | 하차 구역 ID | Integer |
| trip_distance | 이동 거리 (마일) | Float |
| fare_amount | 기본 요금 | Float |
| tip_amount | 팁 (카드 결제만) | Float |
| total_amount | 총 요금 | Float |
| passenger_count | 승객 수 | Integer |
| payment_type | 결제 방식 (1=카드, 2=현금) | Integer |

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# SparkSession 생성
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, count, sum, avg, min, max,
    hour, dayofweek, month, year, to_date,
    round as spark_round, expr, desc, asc,
    lag, lead, row_number, rank, dense_rank,
    unix_timestamp
)
from pyspark.sql.window import Window
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("NYC-Taxi-DataFrame-API") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# 로그 레벨 설정
spark.sparkContext.setLogLevel("WARN")

print("SparkSession 생성 완료!")
print(f"Spark 버전: {spark.version}")

---

## Part 1: 실제 NYC Taxi 데이터 로드

NYC TLC에서 제공하는 실제 Parquet 파일을 로드합니다.

### 데이터 다운로드 스크립트 사용법

```bash
# 기본 실행: 2024년 1월 Yellow Taxi 전체 데이터 다운로드
docker exec -it python-dev python download_data.py

# 샘플링: 50,000건만 추출
docker exec -it python-dev python download_data.py --sample 50000

# 특정 연월 지정
docker exec -it python-dev python download_data.py --year 2023 --month 6

# 택시 타입 변경 (yellow, green, fhv, fhvhv)
docker exec -it python-dev python download_data.py --taxi-type green
```

### 매개변수 설명

| 매개변수 | 기본값 | 설명 |
|----------|--------|------|
| `--year` | 2024 | 데이터 연도 |
| `--month` | 1 | 데이터 월 (1-12) |
| `--taxi-type` | yellow | 택시 종류 (yellow/green/fhv/fhvhv) |
| `--sample` | 0 | 샘플링 건수 (0이면 전체) |

### 데이터셋 재현성

`--sample` 옵션 사용 시 `random_state=42`로 샘플링되어 **동일한 조건이면 항상 같은 데이터셋**이 생성됩니다.
다운로드 실패 시 자동으로 생성되는 샘플 데이터도 `np.random.seed(42)`로 고정되어 있어 재현 가능합니다.

In [ ]:
# -----------------------------------------------------------------------------
# NYC Taxi Parquet 데이터 로드
# -----------------------------------------------------------------------------

# 데이터 파일 경로 (Docker 볼륨 마운트)
DATA_DIR = "/data"

# 사용 가능한 파일 목록 확인
import glob
parquet_files = glob.glob(f"{DATA_DIR}/*.parquet")
print("사용 가능한 데이터 파일:")
for f in parquet_files:
    print(f"  - {os.path.basename(f)}")

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 읽기 (샘플 파일 우선)
# -----------------------------------------------------------------------------

# 샘플 파일이 있으면 우선 사용
sample_files = [f for f in parquet_files if "sample" in f]
if sample_files:
    data_file = sample_files[0]
else:
    data_file = parquet_files[0] if parquet_files else None

if data_file:
    print(f"\n데이터 파일 로드: {data_file}")
    df_taxi = spark.read.parquet(data_file)
    print(f"로드 완료: {df_taxi.count():,}건")
else:
    raise FileNotFoundError(
        "데이터 파일이 없습니다. 먼저 다음 명령으로 데이터를 다운로드하세요:\n"
        "  docker exec -it python-dev python download_data.py --sample 50000"
    )

---

## Part 2: 데이터 탐색 (EDA)

### 스키마 확인

실제 TLC 데이터의 컬럼 구조를 파악합니다.

In [ ]:
# -----------------------------------------------------------------------------
# 스키마 확인
# -----------------------------------------------------------------------------

print("=== NYC Taxi 데이터 스키마 ===")
df_taxi.printSchema()

**예상 출력:**
```
=== NYC Taxi 데이터 스키마 ===
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
```

In [ ]:
# -----------------------------------------------------------------------------
# 샘플 데이터 확인
# -----------------------------------------------------------------------------

# Spark DataFrame을 Pandas로 변환하여 출력 (Jupyter에서 더 깔끔한 표 형태)
print("=== 샘플 데이터 (5건) ===")
df_taxi.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "tip_amount", "total_amount"
).limit(5).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 기술 통계 확인
# -----------------------------------------------------------------------------

print("=== 기술 통계 ===")
df_taxi.select(
    "trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count"
).describe().toPandas()

**예상 출력:**
```
+-------+------------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|   passenger_count|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|             50000|             50000|             50000|             50000|             48813|
|   mean| 3.053276399999998| 15.23834319999999|2.4892406000000004|21.298564200000016|1.3609039026507175|
| stddev|3.9148561438665177|12.672847196879687| 3.150684276419234|15.901155983424348|0.9523499166291498|
|    min|               0.0|            -480.0|             -60.0|            -480.3|               0.0|
|    max|             306.6|             450.0|             380.8|             483.7|               9.0|
+-------+------------------+------------------+------------------+------------------+------------------+
```

**인사이트:** 실제 데이터에는 음수 요금, 이상치가 존재합니다.
데이터 클리닝이 필요한 상황입니다.

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 품질 확인 - 이상치 탐지
# -----------------------------------------------------------------------------

print("=== 데이터 품질 체크 ===")

# 이상치 확인
anomalies = df_taxi.filter(
    (col("fare_amount") < 0) |
    (col("trip_distance") < 0) |
    (col("total_amount") < 0)
).count()

print(f"이상치 (음수 값): {anomalies:,}건")

# NULL 값 확인
print("\n=== NULL 값 비율 ===")
for column in ["passenger_count", "trip_distance", "fare_amount"]:
    null_count = df_taxi.filter(col(column).isNull()).count()
    total = df_taxi.count()
    print(f"{column}: {null_count:,}건 ({null_count/total*100:.2f}%)")

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 클리닝 - 이상치 제거
# -----------------------------------------------------------------------------

print("=== 데이터 클리닝 ===")

df_clean = df_taxi.filter(
    (col("fare_amount") >= 2.5) &           # 최소 기본 요금
    (col("fare_amount") <= 200) &            # 합리적인 최대 요금
    (col("trip_distance") > 0) &
    (col("trip_distance") <= 100) &          # 100마일 이하
    (col("total_amount") > 0) &
    (col("passenger_count") > 0) &
    (col("passenger_count") <= 6)            # 최대 6명
)

original_count = df_taxi.count()
clean_count = df_clean.count()
removed = original_count - clean_count

print(f"원본 데이터: {original_count:,}건")
print(f"클리닝 후: {clean_count:,}건")
print(f"제거된 레코드: {removed:,}건 ({removed/original_count*100:.1f}%)")

# 클리닝된 데이터로 계속 작업
df_taxi = df_clean

---

## Part 3: 데이터 변환 (Transformation)

### 시간 정보 추출

In [ ]:
# -----------------------------------------------------------------------------
# 시간 컬럼 추가
# -----------------------------------------------------------------------------

df_enriched = df_taxi \
    .withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime")) \
    .withColumn("pickup_month", month("tpep_pickup_datetime")) \
    .withColumn(
        "time_of_day",
        when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "Morning")
        .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 18), "Afternoon")
        .when((col("pickup_hour") >= 18) & (col("pickup_hour") < 22), "Evening")
        .otherwise("Night")
    ) \
    .withColumn(
        "is_weekend",
        when(col("pickup_dayofweek").isin(1, 7), True).otherwise(False)
    )

print("=== 시간 정보 추가 결과 ===")
df_enriched.select(
    "tpep_pickup_datetime", "pickup_date", "pickup_hour",
    "time_of_day", "is_weekend"
).limit(5).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 비즈니스 지표 추가
# -----------------------------------------------------------------------------

df_metrics = df_enriched \
    .withColumn(
        "trip_duration_min",
        spark_round(
            (unix_timestamp(col("tpep_dropoff_datetime")) -
             unix_timestamp(col("tpep_pickup_datetime"))) / 60,
            1
        )
    ) \
    .withColumn(
        "fare_per_mile",
        when(col("trip_distance") > 0,
             spark_round(col("fare_amount") / col("trip_distance"), 2))
        .otherwise(0)
    ) \
    .withColumn(
        "tip_percentage",
        when(col("fare_amount") > 0,
             spark_round(col("tip_amount") / col("fare_amount") * 100, 1))
        .otherwise(0)
    )

print("=== 비즈니스 지표 추가 ===")
df_metrics.select(
    "trip_distance", "trip_duration_min",
    "fare_amount", "fare_per_mile",
    "tip_amount", "tip_percentage"
).limit(5).toPandas()

---

## Part 4: 집계 분석

### 시간대별 운행 패턴

In [ ]:
# -----------------------------------------------------------------------------
# 시간대별 통계
# -----------------------------------------------------------------------------

time_stats = df_metrics.groupBy("time_of_day").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("tip_percentage"), 1).alias("avg_tip_pct"),
    spark_round(avg("trip_duration_min"), 1).alias("avg_duration_min")
).orderBy(desc("trip_count"))

print("=== 시간대별 운행 통계 ===")
time_stats.toPandas()

**예상 출력:**
```
+-----------+----------+--------+------------+-----------+----------------+
|time_of_day|trip_count|avg_fare|avg_distance|avg_tip_pct|avg_duration_min|
+-----------+----------+--------+------------+-----------+----------------+
|  Afternoon|     12847|   14.89|        2.85|       18.2|            14.5|
|    Evening|     11234|   15.21|        2.91|       19.1|            15.2|
|    Morning|     10521|   14.45|        2.72|       17.5|            13.8|
|      Night|      8432|   16.12|        3.25|       16.8|            14.1|
+-----------+----------+--------+------------+-----------+----------------+
```

**인사이트:** 오후에 운행이 가장 많고, 저녁에 팁 비율이 가장 높습니다.
야간은 운행이 적지만 평균 거리와 요금이 높습니다.

In [ ]:
# -----------------------------------------------------------------------------
# 시간별 상세 분석 (24시간)
# -----------------------------------------------------------------------------

hourly_stats = df_metrics.groupBy("pickup_hour").agg(
    count("*").alias("trip_count"),
    spark_round(avg("total_amount"), 2).alias("avg_total"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
).orderBy("pickup_hour")

print("=== 시간별 운행 통계 ===")
hourly_stats.toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 주중 vs 주말 비교
# -----------------------------------------------------------------------------

weekend_comparison = df_metrics.groupBy("is_weekend").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("tip_percentage"), 1).alias("avg_tip_pct"),
)

print("=== 주중(false) vs 주말(true) 비교 ===")
weekend_comparison.toPandas()

---

## Part 5: Zone 정보 조인

### TLC Zone Lookup 테이블

NYC TLC는 263개의 택시 구역(Zone)을 정의하고 있습니다.
각 Zone에는 지역명과 행정구역(Borough) 정보가 있습니다.

In [ ]:
# -----------------------------------------------------------------------------
# Zone Lookup 테이블 생성
# -----------------------------------------------------------------------------

# NYC TLC Zone 정보 (주요 지역)
zone_data = [
    (1, "Newark Airport", "EWR"),
    (4, "Alphabet City", "Manhattan"),
    (12, "Battery Park", "Manhattan"),
    (13, "Battery Park City", "Manhattan"),
    (43, "Central Park", "Manhattan"),
    (45, "Chinatown", "Manhattan"),
    (48, "Clinton East", "Manhattan"),
    (68, "East Chelsea", "Manhattan"),
    (79, "East Village", "Manhattan"),
    (87, "Financial District North", "Manhattan"),
    (88, "Financial District South", "Manhattan"),
    (90, "Flatiron", "Manhattan"),
    (100, "Garment District", "Manhattan"),
    (107, "Gramercy", "Manhattan"),
    (113, "Greenwich Village North", "Manhattan"),
    (114, "Greenwich Village South", "Manhattan"),
    (125, "Hudson Sq", "Manhattan"),
    (132, "JFK Airport", "Queens"),
    (138, "LaGuardia Airport", "Queens"),
    (140, "Lenox Hill East", "Manhattan"),
    (141, "Lenox Hill West", "Manhattan"),
    (142, "Lincoln Square East", "Manhattan"),
    (143, "Lincoln Square West", "Manhattan"),
    (144, "Little Italy/NoLiTa", "Manhattan"),
    (148, "Lower East Side", "Manhattan"),
    (151, "Manhattan Valley", "Manhattan"),
    (158, "Meatpacking/West Village West", "Manhattan"),
    (161, "Midtown Center", "Manhattan"),
    (162, "Midtown East", "Manhattan"),
    (163, "Midtown North", "Manhattan"),
    (164, "Midtown South", "Manhattan"),
    (166, "Morningside Heights", "Manhattan"),
    (170, "Murray Hill", "Manhattan"),
    (186, "Penn Station/Madison Sq West", "Manhattan"),
    (209, "Seaport", "Manhattan"),
    (211, "SoHo", "Manhattan"),
    (224, "Stuy Town/PCV", "Manhattan"),
    (230, "Sutton Place/Turtle Bay South", "Manhattan"),
    (231, "Times Sq/Theatre District", "Manhattan"),
    (232, "TriBeCa/Civic Center", "Manhattan"),
    (234, "Union Sq", "Manhattan"),
    (236, "Upper East Side North", "Manhattan"),
    (237, "Upper East Side South", "Manhattan"),
    (238, "Upper West Side North", "Manhattan"),
    (239, "Upper West Side South", "Manhattan"),
    (246, "West Chelsea/Hudson Yards", "Manhattan"),
    (249, "West Village", "Manhattan"),
    (261, "World Trade Center", "Manhattan"),
    (262, "Yorkville East", "Manhattan"),
    (263, "Yorkville West", "Manhattan"),
]

from pyspark.sql.types import StructType, StructField, IntegerType, StringType

zone_schema = StructType([
    StructField("location_id", IntegerType(), True),
    StructField("zone_name", StringType(), True),
    StructField("borough", StringType(), True),
])

df_zones = spark.createDataFrame(zone_data, schema=zone_schema)
print(f"Zone 테이블 생성 완료: {df_zones.count()}개 구역")

In [ ]:
# -----------------------------------------------------------------------------
# 승차 Zone 정보 조인
# -----------------------------------------------------------------------------

df_with_pickup_zone = df_metrics.join(
    df_zones.select(
        col("location_id").alias("PULocationID"),
        col("zone_name").alias("pickup_zone"),
        col("borough").alias("pickup_borough")
    ).withColumn("PULocationID", col("PULocationID").cast("long")),
    "PULocationID",
    "left"
)

print("=== 승차 Zone 조인 결과 ===")
df_with_pickup_zone.select(
    "tpep_pickup_datetime", "PULocationID", "pickup_zone", "pickup_borough"
).filter(col("pickup_zone").isNotNull()).limit(10).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 하차 Zone 정보도 조인
# -----------------------------------------------------------------------------

df_with_zones = df_with_pickup_zone.join(
    df_zones.select(
        col("location_id").alias("DOLocationID"),
        col("zone_name").alias("dropoff_zone"),
        col("borough").alias("dropoff_borough")
    ).withColumn("DOLocationID", col("DOLocationID").cast("long")),
    "DOLocationID",
    "left"
)

print("=== 승하차 Zone 정보 모두 조인 ===")
df_with_zones.select(
    "pickup_zone", "dropoff_zone", "trip_distance", "fare_amount"
).filter(
    col("pickup_zone").isNotNull() & col("dropoff_zone").isNotNull()
).limit(10).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 인기 경로 분석
# -----------------------------------------------------------------------------

popular_routes = df_with_zones \
    .filter(col("pickup_zone").isNotNull() & col("dropoff_zone").isNotNull()) \
    .groupBy("pickup_zone", "dropoff_zone") \
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    ) \
    .orderBy(desc("trip_count"))

print("=== 인기 경로 TOP 15 ===")
popular_routes.limit(15).toPandas()

---

## Part 6: Window Functions

### 시계열 분석

In [ ]:
# -----------------------------------------------------------------------------
# 일별 매출 집계
# -----------------------------------------------------------------------------

daily_revenue = df_metrics.groupBy("pickup_date").agg(
    count("*").alias("trip_count"),
    spark_round(sum("total_amount"), 2).alias("daily_revenue"),
    spark_round(avg("total_amount"), 2).alias("avg_fare"),
).orderBy("pickup_date")

print("=== 일별 매출 ===")
daily_revenue.limit(10).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 누적 매출 계산
# -----------------------------------------------------------------------------

window_cumsum = Window.orderBy("pickup_date").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

daily_with_cumsum = daily_revenue.withColumn(
    "cumulative_revenue",
    spark_round(sum("daily_revenue").over(window_cumsum), 2)
).withColumn(
    "cumulative_trips",
    sum("trip_count").over(window_cumsum)
)

print("=== 일별 누적 매출 ===")
daily_with_cumsum.limit(10).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 7일 이동 평균
# -----------------------------------------------------------------------------

window_ma7 = Window.orderBy("pickup_date").rowsBetween(-6, 0)

daily_with_ma = daily_with_cumsum.withColumn(
    "ma7_revenue",
    spark_round(avg("daily_revenue").over(window_ma7), 2)
).withColumn(
    "ma7_trips",
    spark_round(avg("trip_count").over(window_ma7), 1)
)

print("=== 7일 이동 평균 추가 ===")
daily_with_ma.limit(10).toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 전일 대비 변화율
# -----------------------------------------------------------------------------

window_lag = Window.orderBy("pickup_date")

daily_with_change = daily_with_ma.withColumn(
    "prev_day_revenue",
    lag("daily_revenue", 1).over(window_lag)
).withColumn(
    "revenue_change_pct",
    spark_round(
        (col("daily_revenue") - col("prev_day_revenue")) / col("prev_day_revenue") * 100,
        1
    )
)

print("=== 전일 대비 변화율 ===")
daily_with_change.select(
    "pickup_date", "daily_revenue", "prev_day_revenue", "revenue_change_pct"
).limit(10).toPandas()

---

## Part 7: 공항 노선 분석

JFK(132), LaGuardia(138) 공항 운행을 분석합니다.

In [ ]:
# -----------------------------------------------------------------------------
# 공항 운행 필터링
# -----------------------------------------------------------------------------

AIRPORT_IDS = [132, 138]  # JFK, LaGuardia

# 공항 출발/도착 운행
df_airport = df_metrics.filter(
    (col("PULocationID").isin(AIRPORT_IDS)) |
    (col("DOLocationID").isin(AIRPORT_IDS))
)

airport_count = df_airport.count()
total_count = df_metrics.count()

print(f"=== 공항 관련 운행 ===")
print(f"공항 운행: {airport_count:,}건 ({airport_count/total_count*100:.1f}%)")
print(f"전체 운행: {total_count:,}건")

In [ ]:
# -----------------------------------------------------------------------------
# 공항별 통계
# -----------------------------------------------------------------------------

# 공항 출발 운행
airport_departures = df_metrics.filter(
    col("PULocationID").isin(AIRPORT_IDS)
).withColumn(
    "airport",
    when(col("PULocationID") == 132, "JFK")
    .when(col("PULocationID") == 138, "LaGuardia")
)

airport_stats = airport_departures.groupBy("airport").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("tip_percentage"), 1).alias("avg_tip_pct"),
)

print("=== 공항 출발 운행 통계 ===")
airport_stats.toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 공항 운행 시간대 패턴
# -----------------------------------------------------------------------------

airport_time_pattern = airport_departures.groupBy("airport", "time_of_day").agg(
    count("*").alias("trip_count")
).orderBy("airport", desc("trip_count"))

print("=== 공항별 시간대 패턴 ===")
airport_time_pattern.toPandas()

---

## Part 8: 피벗 테이블

In [ ]:
# -----------------------------------------------------------------------------
# 시간대 x 요일 매트릭스
# -----------------------------------------------------------------------------

df_with_day = df_metrics.withColumn(
    "day_name",
    when(col("pickup_dayofweek") == 1, "Sun")
    .when(col("pickup_dayofweek") == 2, "Mon")
    .when(col("pickup_dayofweek") == 3, "Tue")
    .when(col("pickup_dayofweek") == 4, "Wed")
    .when(col("pickup_dayofweek") == 5, "Thu")
    .when(col("pickup_dayofweek") == 6, "Fri")
    .when(col("pickup_dayofweek") == 7, "Sat")
)

time_day_pivot = df_with_day.groupBy("time_of_day").pivot(
    "day_name",
    ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
).agg(count("*"))

print("=== 시간대 x 요일별 운행 건수 ===")
time_day_pivot.toPandas()

In [ ]:
# -----------------------------------------------------------------------------
# 시간대 x 요일별 평균 요금
# -----------------------------------------------------------------------------

time_day_fare = df_with_day.groupBy("time_of_day").pivot(
    "day_name",
    ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
).agg(spark_round(avg("fare_amount"), 2))

print("=== 시간대 x 요일별 평균 요금 ===")
time_day_fare.toPandas()

---

## Part 9: SQL 연동

In [ ]:
# -----------------------------------------------------------------------------
# SQL 테이블 등록
# -----------------------------------------------------------------------------

df_metrics.createOrReplaceTempView("taxi_trips")
df_zones.createOrReplaceTempView("zones")

print("SQL 테이블 등록 완료: taxi_trips, zones")

In [ ]:
# -----------------------------------------------------------------------------
# SQL로 분석 쿼리
# -----------------------------------------------------------------------------

sql_result = spark.sql("""
    SELECT
        HOUR(tpep_pickup_datetime) as hour,
        z.zone_name,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM taxi_trips t
    LEFT JOIN zones z ON t.PULocationID = z.location_id
    WHERE z.zone_name IS NOT NULL
    GROUP BY HOUR(tpep_pickup_datetime), z.zone_name
    HAVING COUNT(*) >= 10
    ORDER BY trip_count DESC
    LIMIT 20
""")

print("=== SQL: 시간대별 Zone별 분석 TOP 20 ===")
sql_result.toPandas()

---

## 핵심 요약

### 실제 데이터 작업 팁

1. **데이터 품질 체크 필수**: 실제 데이터에는 이상치, NULL, 중복이 있음
2. **클리닝 먼저**: 분석 전 데이터 품질을 확보
3. **샘플링 활용**: 대용량 데이터는 샘플로 프로토타입 개발
4. **toPandas() 활용**: 작은 결과셋은 Pandas로 변환하면 Jupyter에서 더 깔끔하게 출력

### 시간 함수

```python
hour("timestamp_col")       # 시간 추출 (0-23)
dayofweek("timestamp_col")  # 요일 (1=일요일, 7=토요일)
to_date("timestamp_col")    # 날짜만 추출
month("timestamp_col")      # 월 추출 (1-12)
unix_timestamp("timestamp_col")  # Unix timestamp로 변환 (초 단위)
```

### Window Functions

```python
# 누적 합계
Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 이동 평균 (7일)
Window.orderBy("date").rowsBetween(-6, 0)

# 전일 대비
lag("column", 1).over(Window.orderBy("date"))
```

### 조인 패턴

```python
# 컬럼명이 같으면
df1.join(df2, "common_col", "left")

# 컬럼명이 다르면
df1.join(df2, df1.col1 == df2.col2, "left")
```

---

## 실습 과제

### 과제 1: 결제 방식별 분석

payment_type 별로 운행 건수, 평균 요금, 평균 팁 비율을 분석하세요.
(참고: payment_type 1=카드, 2=현금, 3=무료, 4=분쟁)

<details>
<summary>힌트</summary>

```python
df_metrics.groupBy("payment_type").agg(
    count("*").alias("trip_count"),
    # 평균 요금과 팁 비율 추가...
)
```

</details>

<details>
<summary>모범 답안</summary>

```python
payment_analysis = df_metrics.groupBy("payment_type").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("tip_percentage"), 1).alias("avg_tip_pct"),
    spark_round(sum("total_amount"), 2).alias("total_revenue")
).withColumn(
    "payment_name",
    when(col("payment_type") == 1, "Credit Card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No Charge")
    .when(col("payment_type") == 4, "Dispute")
    .otherwise("Unknown")
).orderBy(desc("trip_count"))

payment_analysis.toPandas()
```

</details>

### 과제 2: 장거리 운행 분석

10마일 이상 장거리 운행의 특징을 분석하세요:
- 가장 많이 출발하는 Zone TOP 5
- 평균 운행 시간, 요금, 팁 비율

<details>
<summary>힌트</summary>

```python
df_long_distance = df_metrics.filter(col("trip_distance") >= 10)
```

</details>

<details>
<summary>모범 답안</summary>

```python
# 장거리 운행 필터링
df_long_distance = df_with_zones.filter(col("trip_distance") >= 10)

# 출발 Zone TOP 5
print("=== 장거리 출발 Zone TOP 5 ===")
df_long_distance.filter(col("pickup_zone").isNotNull()) \
    .groupBy("pickup_zone") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(desc("trip_count")) \
    .limit(5).toPandas()

# 장거리 운행 통계
print("=== 장거리 운행 통계 ===")
df_long_distance.agg(
    count("*").alias("total_trips"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("trip_duration_min"), 1).alias("avg_duration_min"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("tip_percentage"), 1).alias("avg_tip_pct")
).toPandas()
```

</details>

### 과제 3: 시간대별 수익성 랭킹

Window Function을 사용하여 각 시간대(time_of_day)에서
pickup_zone별 매출 순위를 계산하세요.

<details>
<summary>힌트</summary>

```python
window_spec = Window.partitionBy("time_of_day").orderBy(desc("total_revenue"))
rank().over(window_spec)
```

</details>

<details>
<summary>모범 답안</summary>

```python
from pyspark.sql.functions import rank

# 시간대 x Zone별 매출 집계
zone_revenue = df_with_zones.filter(col("pickup_zone").isNotNull()) \
    .groupBy("time_of_day", "pickup_zone") \
    .agg(
        count("*").alias("trip_count"),
        spark_round(sum("total_amount"), 2).alias("total_revenue")
    )

# 시간대별 순위 계산
window_spec = Window.partitionBy("time_of_day").orderBy(desc("total_revenue"))

zone_ranking = zone_revenue.withColumn(
    "revenue_rank",
    rank().over(window_spec)
).filter(col("revenue_rank") <= 3)  # TOP 3만

zone_ranking.orderBy("time_of_day", "revenue_rank").toPandas()
```

</details>

---

## 데이터 파이프라인 프로젝트 활용 가이드

이 실습에서 배운 내용은 실제 데이터 파이프라인 구축에 다음과 같이 활용됩니다:

### 1. ETL 파이프라인에서의 활용

```python
# Extract: 다양한 소스에서 데이터 읽기
df = spark.read.parquet("s3://bucket/raw/")      # S3
df = spark.read.jdbc(url, table, properties)     # RDBMS
df = spark.read.json("hdfs:///data/events/")     # HDFS

# Transform: 오늘 배운 변환 기법 적용
df_cleaned = df.filter(...).withColumn(...)
df_enriched = df_cleaned.join(dim_table, ...)
df_aggregated = df_enriched.groupBy(...).agg(...)

# Load: 결과 저장
df_aggregated.write.mode("overwrite").parquet("s3://bucket/processed/")
df_aggregated.write.jdbc(url, "analytics_table", mode="append")
```

### 2. 데이터 품질 체크 자동화

```python
def validate_data(df):
    """데이터 품질 검증 함수"""
    checks = {
        "null_count": df.filter(col("key_column").isNull()).count(),
        "negative_values": df.filter(col("amount") < 0).count(),
        "duplicate_count": df.count() - df.dropDuplicates(["id"]).count()
    }
    return checks
```

### 3. 증분 처리 패턴

```python
# 마지막 처리 시점 이후 데이터만 처리
last_processed = get_checkpoint()  # 메타데이터 테이블에서 조회
df_new = df.filter(col("created_at") > last_processed)

# 처리 후 체크포인트 업데이트
save_checkpoint(current_timestamp)
```

### 4. 파티셔닝 전략

```python
# 날짜별 파티션으로 저장 (쿼리 성능 향상)
df.write \
    .partitionBy("year", "month", "day") \
    .parquet("s3://bucket/partitioned/")
```

In [ ]:
print("\n교시 완료! 다음: Structured Streaming 실습")
spark.stop()

# NYC Taxi 실시간 스트리밍 - Spark Structured Streaming + Kafka

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- Kafka에서 실시간 택시 운행 데이터를 읽을 수 있다
- Window 기반 시간 집계로 실시간 통계를 계산할 수 있다
- Watermark를 사용하여 늦은 데이터를 처리할 수 있다
- 실시간 핫스팟(수요 급증 지역)을 탐지할 수 있다
- 처리 결과를 다양한 Sink로 출력할 수 있다

---

## 실습 환경 안내

이 실습은 Docker 컨테이너(python-dev) 내에서 진행됩니다.

### 실행 방법 1: VSCode Dev Container (권장)

VSCode의 Dev Containers 익스텐션을 사용하면 컨테이너 내부에 직접 접속하여 작업할 수 있습니다.
- 왼쪽 하단에 `python-dev` 또는 컨테이너 이름이 표시됩니다.
- 터미널, 파일 탐색기 모두 컨테이너 내부에서 동작합니다.
- 이 화면에서 바로 Python 파일이나 노트북을 실행하면 됩니다.

### 실행 방법 2: docker exec

로컬 터미널에서 `docker exec` 명령어로 컨테이너 내부 명령을 실행할 수 있습니다.

```bash
# 컨테이너 내부에서 대화형 셸 접속
docker exec -it python-dev bash
```

---

## 실습 아키텍처

![](https://cdn.discordapp.com/attachments/1457516082071081045/1464049294771486834/Gemini_Generated_Image.png?ex=69740da8&is=6972bc28&hm=491e7397d26cc67385ee072a8065b1163e36887da0555350452588fb19abc224&)

**데이터 흐름**

1. **NYC Taxi Data (Parquet)** → producer.py가 실시간으로 Kafka에 전송
2. **Kafka (nyc-taxi-trips)** → 메시지 브로커 역할
3. **Spark Structured Streaming** → 스트림 데이터 처리
4. **출력 (Sink)**
   - Console Sink: 콘솔에 결과 출력
   - Kafka Sink: 처리 결과를 다시 Kafka로 전송
   
---

## 사전 준비

### 1. Docker 환경 시작

```bash
cd docker
docker compose up -d
```

### 2. 데이터 다운로드

```bash
docker exec -it python-dev python download_data.py --sample 10000
```

### 3. Producer 실행 (별도 터미널)

```bash
docker exec -it python-dev python producer.py --rate 10 --duration 120
```

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# SparkSession 생성 (Kafka 연동)
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, from_json, to_json, struct,
    window, count, sum, avg, max, min,
    current_timestamp, expr, desc, when,
    hour, to_timestamp, round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, TimestampType
)
import time

# Kafka 연동을 위한 패키지 (Spark 버전에 맞게)
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8"

spark = SparkSession.builder \
    .appName("NYC-Taxi-Streaming") \
    .master("local[*]") \
    .config("spark.jars.packages", KAFKA_PACKAGE) \
    .config("spark.sql.shuffle.partitions", 4) \
    .config("spark.streaming.stopGracefullyOnShutdown", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("SparkSession 생성 완료!")
print(f"Spark 버전: {spark.version}")

---

## Part 1: Kafka에서 실시간 데이터 읽기

### Kafka 연결 설정

In [ ]:
# -----------------------------------------------------------------------------
# Kafka 연결 설정
# -----------------------------------------------------------------------------

KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "nyc-taxi-trips"

print(f"Kafka 서버: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"토픽: {KAFKA_TOPIC}")

In [ ]:
# -----------------------------------------------------------------------------
# NYC Taxi 메시지 스키마 정의
# -----------------------------------------------------------------------------

# Producer가 전송하는 JSON 메시지 구조
taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("event_time", StringType(), True),  # Producer가 추가하는 이벤트 시간
])

print("택시 메시지 스키마 정의 완료")

In [ ]:
# -----------------------------------------------------------------------------
# Kafka Streaming DataFrame 생성
# -----------------------------------------------------------------------------

# Kafka에서 스트림 읽기
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .load()

print("Kafka 스트림 연결 완료")
print("스키마:")
df_raw.printSchema()

**Kafka DataFrame 스키마:**
```
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)
```

- `key`: 메시지 키 (바이너리)
- `value`: 메시지 본문 (JSON을 바이너리로)
- `timestamp`: Kafka가 기록한 시간

In [ ]:
# -----------------------------------------------------------------------------
# JSON 파싱 및 컬럼 추출
# -----------------------------------------------------------------------------

df_taxi = df_raw \
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_time") \
    .select(
        from_json(col("json_str"), taxi_schema).alias("data"),
        col("kafka_time")
    ) \
    .select(
        col("data.*"),
        col("kafka_time"),
        # event_time을 timestamp로 변환
        to_timestamp(col("data.event_time")).alias("event_timestamp")
    )

print("JSON 파싱 완료")

---

## Part 2: 실시간 집계 - Window 기반

In [ ]:
# -----------------------------------------------------------------------------
# 30초 윈도우 실시간 통계
# -----------------------------------------------------------------------------

# Watermark 설정: 1분 이내 늦은 데이터 허용
realtime_stats = df_taxi \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "30 seconds")
    ) \
    .agg(
        count("*").alias("trip_count"),
        spark_round(sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip")
    )

print("30초 윈도우 집계 정의 완료")

In [ ]:
# -----------------------------------------------------------------------------
# Console Sink로 출력 (테스트용)
# -----------------------------------------------------------------------------

# 스트림 시작 (update 모드: 변경된 결과만 출력)
query_stats = realtime_stats.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="10 seconds") \
    .start()

print("실시간 통계 스트림 시작")
print("Producer를 실행하면 30초 단위 통계가 출력됩니다.")
print("query_stats.stop() 으로 중지")

**예상 출력:**
```
-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+-------------+--------+------------+--------+
|window                                    |trip_count|total_revenue|avg_fare|avg_distance|avg_tip |
+------------------------------------------+----------+-------------+--------+------------+--------+
|{2024-01-15 14:30:00, 2024-01-15 14:30:30}|        12|       284.56|   18.32|        2.85|    3.12|
|{2024-01-15 14:30:30, 2024-01-15 14:31:00}|         8|       186.78|   17.89|        2.41|    2.98|
+------------------------------------------+----------+-------------+--------+------------+--------+
```

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
# 위 스트림이 실행 중일 때, 이 셀을 실행하면 중지됩니다.
# 스트리밍은 백그라운드에서 계속 실행되므로, 필요할 때 직접 중지하세요.
#
# 자동 중지가 필요한 경우 (예: 스크립트 실행):
# time.sleep(60)  # 원하는 시간(초) 동안 대기
query_stats.stop()
print("실시간 통계 스트림 중지됨")

---

## Part 3: 지역별 실시간 수요 분석

In [ ]:
# -----------------------------------------------------------------------------
# 승차 구역별 실시간 수요
# -----------------------------------------------------------------------------

# 1분 윈도우로 지역별 수요 집계
location_demand = df_taxi \
    .withWatermark("event_timestamp", "2 minutes") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("PULocationID")
    ) \
    .agg(
        count("*").alias("pickup_count"),
        spark_round(sum("total_amount"), 2).alias("revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare")
    )

print("지역별 수요 집계 정의 완료")

In [ ]:
# -----------------------------------------------------------------------------
# Memory Sink로 출력 (SQL 쿼리 가능)
# -----------------------------------------------------------------------------

query_location = location_demand.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("location_demand") \
    .trigger(processingTime="15 seconds") \
    .start()

print("지역별 수요 스트림 시작 (메모리 테이블: location_demand)")

In [ ]:
# -----------------------------------------------------------------------------
# 메모리 테이블 쿼리 (실행 중에 수시로 조회 가능)
# -----------------------------------------------------------------------------
# 스트림이 실행 중일 때 이 셀을 반복 실행하여 최신 결과를 확인할 수 있습니다.
# Producer가 데이터를 전송하고 있어야 결과가 나타납니다.

# 상위 수요 지역 조회
print("=== 수요 상위 지역 TOP 10 ===")
spark.sql("""
    SELECT
        PULocationID,
        SUM(pickup_count) as total_pickups,
        ROUND(SUM(revenue), 2) as total_revenue
    FROM location_demand
    GROUP BY PULocationID
    ORDER BY total_pickups DESC
    LIMIT 10
""").show()

**예상 출력:**
```
=== 수요 상위 지역 TOP 10 ===
+------------+-------------+-------------+
|PULocationID|total_pickups|total_revenue|
+------------+-------------+-------------+
|         161|           23|       456.78|
|         237|           19|       378.45|
|         162|           17|       345.23|
|         132|           15|       892.10|
|         138|           14|       756.34|
+------------+-------------+-------------+
```

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
query_location.stop()
print("지역별 수요 스트림 중지됨")

---

## Part 4: 핫스팟 탐지 (수요 급증 알림)

In [ ]:
# -----------------------------------------------------------------------------
# 핫스팟 탐지: 30초 내 5건 이상 승차 지역
# -----------------------------------------------------------------------------

HOTSPOT_THRESHOLD = 5  # 30초 내 이 건수 이상이면 핫스팟

hotspot_detection = df_taxi \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "30 seconds", "15 seconds"),  # 15초 슬라이딩
        col("PULocationID")
    ) \
    .agg(
        count("*").alias("pickup_count")
    ) \
    .filter(col("pickup_count") >= HOTSPOT_THRESHOLD)

print(f"핫스팟 탐지 정의 완료 (임계값: {HOTSPOT_THRESHOLD}건/30초)")

In [ ]:
# -----------------------------------------------------------------------------
# 핫스팟 알림 출력
# -----------------------------------------------------------------------------

def alert_hotspot(df, epoch_id):
    """핫스팟 발견 시 알림"""
    if df.count() > 0:
        print(f"\n{'='*60}")
        print(f"[HOTSPOT ALERT] Epoch {epoch_id}")
        print(f"{'='*60}")
        df.show(truncate=False)

query_hotspot = hotspot_detection.writeStream \
    .outputMode("update") \
    .foreachBatch(alert_hotspot) \
    .trigger(processingTime="15 seconds") \
    .start()

print("핫스팟 탐지 스트림 시작")
print(f"30초 내 {HOTSPOT_THRESHOLD}건 이상 승차가 발생하면 알림 출력")

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
query_hotspot.stop()
print("핫스팟 탐지 스트림 중지됨")

**예상 출력:**
```
============================================================
[HOTSPOT ALERT] Epoch 3
============================================================
+------------------------------------------+------------+------------+
|window                                    |PULocationID|pickup_count|
+------------------------------------------+------------+------------+
|{2024-01-15 14:32:00, 2024-01-15 14:32:30}|         161|           7|
|{2024-01-15 14:32:00, 2024-01-15 14:32:30}|         237|           5|
+------------------------------------------+------------+------------+
```

**인사이트:** Zone 161(Midtown Center), 237(Upper East Side South)에서
수요가 급증하고 있습니다.

---

## Part 5: ForeachBatch - 배치별 커스텀 처리

In [ ]:
# -----------------------------------------------------------------------------
# foreachBatch: 각 마이크로배치에 커스텀 로직 적용
# -----------------------------------------------------------------------------

def process_batch(batch_df, batch_id):
    """각 마이크로배치를 처리하는 함수"""
    if batch_df.count() == 0:
        print(f"Batch {batch_id}: 데이터 없음")
        return

    print(f"\n{'='*60}")
    print(f"Batch {batch_id} 처리 중...")
    print(f"{'='*60}")

    # 배치 통계
    stats = batch_df.agg(
        count("*").alias("total_trips"),
        spark_round(sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(max("tip_amount"), 2).alias("max_tip")
    ).collect()[0]

    print(f"  총 운행: {stats['total_trips']}건")
    print(f"  총 매출: ${stats['total_revenue']}")
    print(f"  평균 요금: ${stats['avg_fare']}")
    print(f"  최대 팁: ${stats['max_tip']}")

    # 시간대별 분포
    print("\n  상위 승차 지역:")
    batch_df.groupBy("PULocationID") \
        .count() \
        .orderBy(desc("count")) \
        .show(5, truncate=False)


# foreachBatch 스트림
query_batch = df_taxi.writeStream \
    .outputMode("append") \
    .foreachBatch(process_batch) \
    .trigger(processingTime="30 seconds") \
    .start()

print("foreachBatch 스트림 시작")

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
query_batch.stop()
print("foreachBatch 스트림 중지됨")

**예상 출력:**
```
============================================================
Batch 2 처리 중...
============================================================
  총 운행: 287건
  총 매출: $5234.67
  평균 요금: $15.32
  최대 팁: $28.50

  상위 승차 지역:
+------------+-----+
|PULocationID|count|
+------------+-----+
|161         |23   |
|237         |19   |
|162         |17   |
|132         |15   |
|138         |12   |
+------------+-----+
```

---

## Part 6: Kafka Sink - 처리 결과를 Kafka로 전송

In [ ]:
# -----------------------------------------------------------------------------
# 집계 결과를 다른 Kafka 토픽으로 전송
# -----------------------------------------------------------------------------

OUTPUT_TOPIC = "taxi-analytics"

# 실시간 통계를 JSON으로 변환
analytics_output = realtime_stats \
    .select(
        to_json(struct(
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("trip_count"),
            col("total_revenue"),
            col("avg_fare"),
            col("avg_distance")
        )).alias("value")
    )

# Kafka로 전송
query_kafka_sink = analytics_output.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("topic", OUTPUT_TOPIC) \
    .option("checkpointLocation", "/tmp/checkpoint-analytics") \
    .outputMode("update") \
    .start()

print(f"분석 결과를 {OUTPUT_TOPIC} 토픽으로 전송 중")

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
query_kafka_sink.stop()
print("Kafka Sink 스트림 중지됨")

---

## Part 7: 공항 노선 실시간 모니터링

In [ ]:
# -----------------------------------------------------------------------------
# 공항 관련 운행 실시간 추적
# -----------------------------------------------------------------------------

AIRPORT_IDS = [132, 138]  # JFK, LaGuardia

# 공항 출발/도착 필터
df_airport = df_taxi.filter(
    (col("PULocationID").isin(AIRPORT_IDS)) |
    (col("DOLocationID").isin(AIRPORT_IDS))
).withColumn(
    "airport_direction",
    when(col("PULocationID").isin(AIRPORT_IDS), "FROM_AIRPORT")
    .otherwise("TO_AIRPORT")
).withColumn(
    "airport",
    when(col("PULocationID") == 132, "JFK")
    .when(col("PULocationID") == 138, "LaGuardia")
    .when(col("DOLocationID") == 132, "JFK")
    .when(col("DOLocationID") == 138, "LaGuardia")
)

# 1분 윈도우 공항별 통계
airport_stats = df_airport \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("airport"),
        col("airport_direction")
    ) \
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance")
    )

query_airport = airport_stats.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="30 seconds") \
    .start()

print("공항 노선 모니터링 스트림 시작")

In [ ]:
# -----------------------------------------------------------------------------
# 스트림 중지 (수동 실행)
# -----------------------------------------------------------------------------
query_airport.stop()
print("공항 노선 스트림 중지됨")

**예상 출력:**
```
+------------------------------------------+---------+----------------+----------+--------+------------+
|window                                    |airport  |airport_direction|trip_count|avg_fare|avg_distance|
+------------------------------------------+---------+----------------+----------+--------+------------+
|{2024-01-15 14:35:00, 2024-01-15 14:36:00}|JFK      |FROM_AIRPORT    |         5|   52.30|       15.23|
|{2024-01-15 14:35:00, 2024-01-15 14:36:00}|LaGuardia|TO_AIRPORT      |         3|   35.45|        8.92|
|{2024-01-15 14:35:00, 2024-01-15 14:36:00}|JFK      |TO_AIRPORT      |         4|   48.75|       14.56|
+------------------------------------------+---------+----------------+----------+--------+------------+
```

---

## Part 8: 스트림 상태 모니터링

In [ ]:
# -----------------------------------------------------------------------------
# 활성 스트림 확인
# -----------------------------------------------------------------------------

print("=== 활성 스트림 목록 ===")
for stream in spark.streams.active:
    print(f"  - {stream.name or stream.id}: {stream.status}")

In [ ]:
# -----------------------------------------------------------------------------
# 모든 스트림 중지 함수
# -----------------------------------------------------------------------------

def stop_all_streams():
    """모든 활성 스트림 중지"""
    for stream in spark.streams.active:
        print(f"중지 중: {stream.name or stream.id}")
        stream.stop()
    print("모든 스트림 중지 완료")

# 필요시 실행
# stop_all_streams()

---

## 실습 과제

### 과제 1: 결제 방식별 실시간 통계

payment_type별로 실시간 통계를 집계하세요:
- 1분 윈도우 기준
- 결제 방식별 건수, 평균 요금, 평균 팁

<details>
<summary>힌트</summary>

```python
payment_stats = df_taxi \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("payment_type")
    ) \
    .agg(...)
```

</details>

<details>
<summary>모범 답안</summary>

```python
payment_stats = df_taxi \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("payment_type")
    ) \
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip"),
        spark_round(sum("total_amount"), 2).alias("total_revenue")
    )

query_payment = payment_stats.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="30 seconds") \
    .start()

# 확인 후 중지
# query_payment.stop()
```

</details>

### 과제 2: 팁 비율 이상 탐지

팁 비율이 비정상적으로 높은(50% 이상) 운행을 실시간으로 탐지하세요.

<details>
<summary>힌트</summary>

```python
df_taxi \
    .withColumn("tip_ratio", col("tip_amount") / col("fare_amount") * 100) \
    .filter(col("tip_ratio") >= 50)
```

</details>

<details>
<summary>모범 답안</summary>

```python
# 팁 비율 계산 및 이상치 필터
df_high_tip = df_taxi \
    .withColumn(
        "tip_ratio",
        when(col("fare_amount") > 0,
             spark_round(col("tip_amount") / col("fare_amount") * 100, 1))
        .otherwise(0)
    ) \
    .filter(col("tip_ratio") >= 50) \
    .select(
        "event_timestamp", "PULocationID", "DOLocationID",
        "fare_amount", "tip_amount", "tip_ratio"
    )

def alert_high_tip(df, epoch_id):
    if df.count() > 0:
        print(f"\n[HIGH TIP ALERT] Epoch {epoch_id}")
        df.show(truncate=False)

query_high_tip = df_high_tip.writeStream \
    .outputMode("append") \
    .foreachBatch(alert_high_tip) \
    .trigger(processingTime="15 seconds") \
    .start()

# 확인 후 중지
# query_high_tip.stop()
```

</details>

### 과제 3: 실시간 대시보드 데이터

Memory Sink를 사용하여 다음을 조회할 수 있는 테이블을 만드세요:
- 분당 총 운행 건수
- 분당 총 매출
- 가장 인기 있는 승차 지역 TOP 5

<details>
<summary>힌트</summary>

```python
.format("memory")
.queryName("dashboard_data")

spark.sql("SELECT * FROM dashboard_data")
```

</details>

<details>
<summary>모범 답안</summary>

```python
# 대시보드용 집계 데이터
dashboard_data = df_taxi \
    .withWatermark("event_timestamp", "2 minutes") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("PULocationID")
    ) \
    .agg(
        count("*").alias("trip_count"),
        spark_round(sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare")
    )

query_dashboard = dashboard_data.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("dashboard_data") \
    .trigger(processingTime="30 seconds") \
    .start()

# 조회 쿼리 (스트림 실행 중 반복 실행 가능)
spark.sql("""
    SELECT
        SUM(trip_count) as total_trips,
        ROUND(SUM(total_revenue), 2) as total_revenue
    FROM dashboard_data
""").show()

# 인기 지역 TOP 5
spark.sql("""
    SELECT PULocationID, SUM(trip_count) as pickups
    FROM dashboard_data
    GROUP BY PULocationID
    ORDER BY pickups DESC
    LIMIT 5
""").show()

# 확인 후 중지
# query_dashboard.stop()
```

</details>

---

## 핵심 요약

### Kafka Source

```python
spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "topic-name") \
    .option("startingOffsets", "latest") \
    .load()
```

### JSON 파싱

```python
# Kafka value는 binary이므로 string으로 변환 후 파싱
df.selectExpr("CAST(value AS STRING)") \
  .select(from_json(col("value"), schema).alias("data")) \
  .select("data.*")
```

### Window 집계

```python
# 고정 윈도우
window(col("timestamp"), "1 minute")

# 슬라이딩 윈도우 (30초 윈도우, 15초 슬라이드)
window(col("timestamp"), "30 seconds", "15 seconds")
```

### Watermark

```python
df.withWatermark("event_time", "1 minute")
```

### Output Modes

| 모드 | 설명 | 사용 사례 |
|------|------|----------|
| append | 새 행만 출력 | 필터링, 맵핑 |
| update | 변경된 행만 출력 | 집계 (최신 결과) |
| complete | 전체 결과 출력 | 집계 (전체 필요시) |

### Sinks

```python
# Console (디버깅)
.format("console")

# Memory (SQL 쿼리용)
.format("memory").queryName("table_name")

# Kafka (다른 토픽으로)
.format("kafka").option("topic", "output-topic")

# foreachBatch (커스텀 처리)
.foreachBatch(process_function)
```

---

## 다양한 소스 및 싱크 활용 가이드

### 지원되는 Source (입력)

| Source | 설명 | 예시 |
|--------|------|------|
| Kafka | 실시간 메시지 스트림 | `.format("kafka").option("subscribe", "topic")` |
| File | 디렉토리 모니터링 (JSON, CSV, Parquet) | `.format("json").load("/data/incoming/")` |
| Socket | TCP 소켓 (테스트용) | `.format("socket").option("host", "localhost").option("port", 9999)` |
| Rate | 테스트용 더미 데이터 생성 | `.format("rate").option("rowsPerSecond", 10)` |

### 지원되는 Sink (출력)

| Sink | 설명 | 사용 사례 |
|------|------|----------|
| Console | 터미널 출력 | 개발/디버깅 |
| Memory | 메모리 테이블 | SQL 쿼리, 대시보드 |
| Kafka | Kafka 토픽 전송 | 다운스트림 처리 |
| File (Parquet/JSON) | 파일 저장 | 데이터 레이크 적재 |
| foreachBatch | 커스텀 처리 | DB 저장, API 호출 |
| foreach | 레코드별 처리 | 세밀한 제어 |

### 처리 결과 확인 방법

#### 1. Console Sink 확인
```python
# 스트림 시작 후 터미널에서 출력 확인
query = df.writeStream.format("console").start()
```

#### 2. Memory Sink 조회
```python
# 메모리 테이블에 저장 후 SQL로 조회
query = df.writeStream.format("memory").queryName("my_table").start()
spark.sql("SELECT * FROM my_table").show()
```

#### 3. Kafka 출력 확인
```bash
# 별도 터미널에서 kafka-console-consumer 실행
docker exec -it kafka kafka-console-consumer \
    --bootstrap-server localhost:9092 \
    --topic output-topic \
    --from-beginning
```

#### 4. 파일 출력 확인
```bash
# 출력 디렉토리에서 Parquet 파일 확인
ls -la /data/output/
```

---

## 데이터 파이프라인 프로젝트 활용 가이드

이 실습에서 배운 내용은 실제 데이터 파이프라인 구축에 다음과 같이 활용됩니다:

### 1. 실시간 데이터 수집 파이프라인

```python
# 실시간 이벤트 → Kafka → Spark Streaming → 데이터 레이크
df = spark.readStream.format("kafka") \
    .option("subscribe", "raw-events").load()

# 변환 및 정제
df_cleaned = df.select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .filter(col("event_type").isNotNull())

# Parquet로 저장 (파티션별)
df_cleaned.writeStream \
    .format("parquet") \
    .option("path", "s3://bucket/events/") \
    .option("checkpointLocation", "/checkpoint/events") \
    .partitionBy("date", "hour") \
    .start()
```

### 2. 실시간 알림 시스템

```python
def send_alert(df, epoch_id):
    """이상 탐지 시 알림 전송"""
    for row in df.collect():
        # Slack, Email, SMS 등으로 알림
        send_slack_message(f"Alert: {row}")

df_anomalies.writeStream \
    .foreachBatch(send_alert) \
    .start()
```

### 3. 실시간 대시보드 연동

```python
# Redis, Elasticsearch 등에 실시간 저장
def update_dashboard(df, epoch_id):
    # Pandas로 변환 후 외부 시스템에 저장
    pdf = df.toPandas()
    redis_client.set("realtime_stats", pdf.to_json())

df_stats.writeStream \
    .foreachBatch(update_dashboard) \
    .trigger(processingTime="10 seconds") \
    .start()
```

### 4. 장애 복구 및 체크포인트

```python
# 체크포인트로 장애 복구 보장
query = df.writeStream \
    .option("checkpointLocation", "/checkpoint/my-job") \
    .start()

# 재시작 시 마지막 처리 지점부터 재개
```

### 5. 모니터링 및 운영

```python
# 스트림 상태 확인
print(query.status)          # 현재 상태
print(query.recentProgress)  # 최근 처리 통계
print(query.lastProgress)    # 마지막 배치 정보

# 처리량 모니터링
progress = query.lastProgress
if progress:
    print(f"Input rows/sec: {progress['inputRowsPerSecond']}")
    print(f"Processed rows/sec: {progress['processedRowsPerSecond']}")
```

In [ ]:
# 세션 정리
print("\n스트림 정리 후 세션 종료")
stop_all_streams()
spark.stop()
print("완료!")